# Option A — Walkthrough of every code change

This notebook explains, step by step, every modification that turns the
vanilla `Deformable-3D-Gaussians` codebase into the **Option A** variant:

> Split Gaussians into **human** and **background** sets. Only human
> Gaussians are routed through the time-conditioned deformation MLP;
> background Gaussians stay fully static.

For each change we give, in this order:

1. **Goal** of the change (≤ 2 sentences).
2. **Exact code** that was added / modified, with its `# Changed — Option A`
   marker.
3. **Theory** you need to understand *what* the snippet does and *why*.

All cells are markdown — nothing here executes. Treat the notebook as a
commented diff against the upstream codebase.

### Data flow recap (0.1-minute version)

```
NeuMan scene on disk
   ├── sparse/           ── COLMAP cameras + SfM points (static background)
   ├── images/           ── RGB frames
   ├── segmentations/    ── binary human masks (per frame)
   └── depth_maps/       ── COLMAP-MVS geometric depth (.geometric.bin)

dataset_readers.py  ─▶  BasicPointCloud(points, colors, is_human∈{0,1})
                                │
scene/__init__.py   ─▶  Scene  │  (dispatch to NeuMan reader)
                                ▼
gaussian_model.py   ─▶  GaussianModel  ─ tag `_is_human` alongside _xyz
                                │
deform_model.py     ─▶  step(xyz, t, is_human)  ── MLP runs on humans only
                                ▼
train.py / render.py / train_gui.py  ── pass `is_human=` at each step
```

The next sections walk this pipeline from top to bottom.


---
## 1. `utils/graphics_utils.py` — Add the `is_human` tag to `BasicPointCloud`

### 1.1 Goal
Give the initial point cloud an optional per-point tag so the reader can say
"this point is a human seed" or "this point is a background SfM point".
Everything downstream (the GaussianModel, checkpointing, …) then carries
that tag.

### 1.2 Code (lines 18–26)

```python
# Changed — Option A: BasicPointCloud carries an optional per-point is_human
# tag so the NeuMan reader can mark human-backprojected seeds (True) separately
# from background COLMAP points (False). Legacy readers leave it as None, which
# downstream code interprets as "no tag / route all Gaussians through the MLP".
class BasicPointCloud(NamedTuple):
    points: np.array
    colors: np.array
    normals: np.array
    is_human: np.array = None  # Changed — Option A
```

### 1.3 Theory

- **`NamedTuple`** gives an immutable struct with typed fields. Adding a
  field *with a default value* (`= None`) is a non-breaking change: any
  caller that constructs `BasicPointCloud(points, colors, normals)` still
  works.
- **Why `None` as the sentinel?** An empty array (`np.empty(0)`) would
  collide with the legitimate "zero points" case. `None` means *"no
  information"*, and downstream code uses `pcd.is_human is not None` to
  decide whether a semantic split is available.
- **Why here and not as a parallel dict?** The initial cloud is passed
  through several layers (reader → `Scene` → `GaussianModel.create_from_pcd`).
  Putting the tag in the same container guarantees ordering stays aligned
  with `points` — parallel arrays in different objects drift over time.


---
## 2. `scene/dataset_readers.py` — Read NeuMan and build the tagged cloud

The reader does the heavy lifting: find COLMAP data, load depth & masks,
backproject masked pixels to 3D, and return a `BasicPointCloud` whose points
are correctly tagged.

Six related additions live in this file; we'll take them in dataflow order.


### 2.1 Handle NeuMan's `sparse/` layout (helper `_find_sparse_dir`)

**Goal.** The upstream 3DGS reader hard-codes `sparse/0/` (the default
COLMAP output). NeuMan ships COLMAP files flat under `sparse/`. This helper
tries both, so the same reader works with either layout.

**Code (lines 173–184, and 188 where it is used):**

```python
# Changed — Option A: NeuMan scenes ship COLMAP files under <scene>/sparse/
# (not <scene>/sparse/0/ like the original 3DGS layout). This helper finds the
# correct directory without forcing the user to symlink.
def _find_sparse_dir(path):
    """Return the subdirectory holding cameras.{bin,txt}, checking sparse/0/
    first (vanilla 3DGS) then sparse/ (NeuMan). Raises if neither exists."""
    for candidate in (os.path.join(path, "sparse", "0"),
                      os.path.join(path, "sparse")):
        if os.path.isfile(os.path.join(candidate, "cameras.bin")) or \
           os.path.isfile(os.path.join(candidate, "cameras.txt")):
            return candidate
    raise FileNotFoundError(f"No COLMAP sparse directory under {path}")


def readColmapSceneInfo(path, images, eval, llffhold=8):
    sparse_dir = _find_sparse_dir(path)  # Changed — Option A: sparse/ fallback
    ...
```

**Theory.** COLMAP's sparse reconstruction output layout is
`<project>/sparse/<model-index>/` where `<model-index>` is usually `0` when
there is a single model. NeuMan pre-processes its data and stores only a
single reconstruction directly under `sparse/`. Robust readers probe
multiple candidates rather than fail on a cosmetic path difference.


### 2.2 Read COLMAP-MVS depth maps (`_read_colmap_depth_bin`)

**Goal.** Parse COLMAP's undocumented `.geometric.bin` depth format so we
can recover per-pixel scene depth for the frames we want to backproject.

**Code (lines 616–627):**

```python
# Changed — Option A: read a COLMAP MVS geometric depth map. The file format
# is a short ASCII header "width&height&channels&" followed by raw float32
# little-endian data. Returns a (H, W) np.float32 array (channels=1).
def _read_colmap_depth_bin(path):
    with open(path, 'rb') as f:
        data = f.read()
    m = re.match(rb'(\d+)&(\d+)&(\d+)&', data)
    if m is None:
        raise ValueError(f"Unknown COLMAP depth format: {path}")
    w, h, c = int(m.group(1)), int(m.group(2)), int(m.group(3))
    depth = np.frombuffer(data[m.end():], dtype=np.float32).reshape(h, w, c)
    return depth[..., 0] if c == 1 else depth
```

**Theory.**

- **COLMAP-MVS** runs after sparse SfM: it estimates a dense *depth* and
  *normal* map per input image using PatchMatch stereo. The
  `<frame>.geometric.bin` file stores the geometric-consistency-refined
  depth (as opposed to `.photometric.bin`, which is the raw photometric
  result — noisier).
- **File layout.** A short ASCII header `W&H&C&` (no line break) followed
  by `W*H*C` little-endian `float32` values in row-major (H, W, C) order.
  The `&` separator is a COLMAP-specific quirk. We use a byte-regex
  because the header length varies with W/H magnitudes.
- **Depth units.** In metric reconstructions this is meters; in
  unit-normalized NeuMan scenes it's the same scale as the COLMAP point
  cloud — safe to mix with the camera extrinsics.


### 2.3 Read NeuMan's binary human mask (`_read_human_mask`)

**Goal.** Load the segmentation PNG and normalize it to a clean boolean
array `(H, W)` aligned with the depth-map resolution.

**Code (lines 630–640):**

```python
# Changed — Option A: build a binary human mask from a NeuMan segmentation PNG.
# Any non-zero pixel in the first channel is considered "inside the human".
def _read_human_mask(path, target_hw):
    mask = np.array(Image.open(path))
    if mask.ndim == 3:
        mask = mask[..., 0]
    if mask.shape != target_hw:
        # Resize with nearest-neighbor to match the camera/depth resolution.
        mask = cv.resize(mask, (target_hw[1], target_hw[0]),
                         interpolation=cv.INTER_NEAREST)
    return mask > 0
```

**Theory.**

- **NeuMan masks** are binary PNGs produced by an off-the-shelf human
  segmentor (e.g. PointRend). They are sometimes saved as RGB or RGBA
  where every channel holds the same bit — `mask[..., 0]` safely collapses
  to a single plane.
- **Nearest-neighbor resize.** Bilinear would introduce half-pixel values
  and blur the boundary, turning "human"/"background" into a soft mix.
  Segmentation masks must stay strictly binary, so we use
  `cv.INTER_NEAREST`.
- **Resolution alignment.** The depth map (`H_d, W_d`) and the image
  (`H_i, W_i`) sometimes differ slightly. We resize the mask to the depth
  grid because that's where we'll index into `depth[v, u]` next.


### 2.4 Backproject pixels into world space (`_backproject_pixels`)

**Goal.** Given a pinhole camera with intrinsics `(fx, fy, cx, cy)` and
extrinsics `(R, T)` stored in the `CameraInfo` convention, lift a batch of
pixel coordinates `(u, v)` at depth `d` from image space into world space.

**Code (lines 643–654):**

```python
# Changed — Option A: backproject a camera-space pixel grid into world space.
# Uses stored CameraInfo convention (R = C2W rotation, T = world-to-cam
# translation in camera frame), matching readColmapCameras above.
def _backproject_pixels(u, v, depth_values, fx, fy, cx, cy, R, T):
    x_cam = (u - cx) / fx * depth_values
    y_cam = (v - cy) / fy * depth_values
    z_cam = depth_values
    X_cam = np.stack([x_cam, y_cam, z_cam], axis=-1)  # (N, 3)
    # CameraInfo stores R = R_cm^T and T = t_cm (see readColmapCameras). So
    # World-from-camera: X_world = R @ (X_cam - T).
    X_world = (X_cam - T[None, :]) @ R.T
    return X_world
```

**Theory.**

- **Pinhole inverse projection.** For a pixel `(u, v)` with depth `d`,
  the camera-space 3D point is

  $$
  X_{\text{cam}} =
  \begin{pmatrix}(u - c_x)\, d / f_x \\\\ (v - c_y)\, d / f_y \\\\ d\end{pmatrix}.
  $$

  This just inverts the pinhole projection
  `u = fx * X/Z + cx, v = fy * Y/Z + cy`.

- **Camera convention matters.** In this codebase, the `CameraInfo` class
  stores `R` as the transpose of the world→camera rotation (i.e. the
  *camera-to-world* rotation), and `T` as the **world-space coordinates of
  the camera origin expressed in the camera frame** (see
  `readColmapCameras`). The correct world-from-camera formula is therefore

  $$ X_{\text{world}} = R\,(X_{\text{cam}} - T). $$

  Writing it in NumPy with row-vectors gives `(X_cam - T) @ R.T`, which is
  exactly the line in the code.

- **Why not use the 4×4 world-view matrix?** We could — but the
  CameraInfo stores `R, T` directly, and this keeps the per-frame cost low
  (one matmul per batch of pixels, no inversions).


### 2.5 Sample human-tagged seed points (`_seed_human_points_from_masks`)

**Goal.** For a handful of evenly spaced training frames, turn (mask ∧
valid-depth) pixels into 3D human seed points with matching colors. This
is what actually produces the "human cloud" at initialization.

**Code (lines 657–723, slightly compressed comments):**

```python
# Changed — Option A: sample human-tagged seed points by backprojecting masked
# depth maps for a subset of frames. Returns (xyz, rgb) arrays or (None, None)
# if no mask/depth pairs were usable.
def _seed_human_points_from_masks(cam_infos, source_path,
                                  mask_dir="segmentations",
                                  depth_dir="depth_maps",
                                  depth_suffix=".geometric.bin",
                                  n_frames=8,
                                  max_points_per_frame=5000,
                                  depth_min=0.1, depth_max=50.0):
    if len(cam_infos) == 0:
        return None, None
    step = max(1, len(cam_infos) // n_frames)
    selected = cam_infos[::step][:n_frames]

    all_xyz, all_rgb = [], []
    for cam in selected:
        mask_path = os.path.join(source_path, mask_dir, cam.image_name + ".png")
        depth_path = os.path.join(source_path, depth_dir,
                                  cam.image_name + ".png" + depth_suffix)
        if not (os.path.isfile(mask_path) and os.path.isfile(depth_path)):
            continue

        depth = _read_colmap_depth_bin(depth_path)
        H, W = depth.shape
        mask = _read_human_mask(mask_path, (H, W))
        valid = mask & (depth > depth_min) & (depth < depth_max)
        if valid.sum() == 0:
            continue

        # Intrinsics from FOV + image size (NeuMan pinholes have cx=W/2, cy=H/2
        # — confirmed from sparse/cameras.txt).
        fx = fov2focal(cam.FovX, W)
        fy = fov2focal(cam.FovY, H)
        cx, cy = W / 2.0, H / 2.0

        vs, us = np.where(valid)
        n_valid = len(us)
        if n_valid > max_points_per_frame:
            idx = np.random.choice(n_valid, max_points_per_frame, replace=False)
            us, vs = us[idx], vs[idx]
        d_vals = depth[vs, us]

        X_world = _backproject_pixels(us.astype(np.float32),
                                      vs.astype(np.float32),
                                      d_vals.astype(np.float32),
                                      fx, fy, cx, cy, cam.R, cam.T)

        rgb = np.array(cam.image).astype(np.float32) / 255.0
        if rgb.ndim == 2:
            rgb = np.repeat(rgb[..., None], 3, axis=-1)
        if rgb.shape[:2] != (H, W):
            rgb = cv.resize(rgb, (W, H), interpolation=cv.INTER_LINEAR)
        colors = rgb[vs, us, :3]

        all_xyz.append(X_world.astype(np.float32))
        all_rgb.append(colors.astype(np.float32))

    if len(all_xyz) == 0:
        return None, None
    return np.concatenate(all_xyz, axis=0), np.concatenate(all_rgb, axis=0)
```

**Theory and key design choices.**

1. **Why only 8 frames, not all?** The subject in a monocular video moves;
   backprojecting *every* frame would produce a thick, blurry human cloud
   (the same limb appears at many positions). Using a sparse set ensures
   seeds are roughly in a single canonical pose region; densification +
   the deformation MLP take care of the rest during training.

2. **Why subsample to 5k points/frame?** NeuMan images are ~1080p → a
   human mask can contain 300k+ pixels. Without subsampling the human
   cloud would dominate initialization (>2M points) and starve the
   densifier of budget for the background. 5k × 8 ≈ 40k points is a good
   compromise: enough for the human to emerge quickly, small enough that
   densification still has headroom.

3. **Why `fov2focal(FovX, W)` for `fx`?** Focal length in pixels relates
   to field-of-view and image width via `fx = W / (2 tan(FovX/2))`. This
   codebase already provides `fov2focal` in `utils.graphics_utils`.

4. **`cx = W/2`, `cy = H/2`.** NeuMan uses an idealized `PINHOLE` camera
   model with principal point at the image center. If you adapt this to a
   non-NeuMan dataset with off-center principal points, read them from
   `cam.params[2], cam.params[3]` instead.

5. **Depth gating `(depth > 0.1) & (depth < 50)`.** COLMAP-MVS writes `0`
   for unreconstructed pixels and can produce arbitrarily large outliers.
   The range brackets out bad samples before they become wild-positioned
   seed Gaussians.


### 2.6 The NeuMan scene reader (`readNeuManSceneInfo`) + callback registration

**Goal.** Thin wrapper that glues everything together: run the COLMAP
reader first, append the human seeds, build the tagged `BasicPointCloud`,
and save a traceability PLY. Also register the reader under the name
`"NeuMan"` in the dispatch table.

**Code (lines 726–768, 777):**

```python
# Changed — Option A: NeuMan scene reader. Reuses the COLMAP reader for
# cameras and the background PLY, then appends human seed points obtained by
# backprojecting masked depth maps. The returned BasicPointCloud carries an
# is_human tag per point (False for COLMAP background, True for human seeds).
def readNeuManSceneInfo(path, images, eval, llffhold=8):
    base = readColmapSceneInfo(path, images, eval, llffhold=llffhold)

    bg_points = np.asarray(base.point_cloud.points) if base.point_cloud is not None else np.zeros((0, 3), np.float32)
    bg_colors = np.asarray(base.point_cloud.colors) if base.point_cloud is not None else np.zeros((0, 3), np.float32)

    human_xyz, human_rgb = _seed_human_points_from_masks(
        base.train_cameras, path)
    if human_xyz is None or len(human_xyz) == 0:
        print("[Option A] No human seed points found — falling back to "
              "background-only initialization.")
        human_xyz = np.zeros((0, 3), np.float32)
        human_rgb = np.zeros((0, 3), np.float32)
    else:
        print(f"[Option A] Seeded {len(human_xyz)} human points from "
              f"{min(8, len(base.train_cameras))} frames.")

    points = np.concatenate([bg_points, human_xyz], axis=0).astype(np.float32)
    colors = np.concatenate([bg_colors, human_rgb], axis=0).astype(np.float32)
    normals = np.zeros_like(points, dtype=np.float32)
    is_human = np.concatenate([
        np.zeros(len(bg_points), dtype=bool),
        np.ones(len(human_xyz), dtype=bool),
    ], axis=0)

    pcd = BasicPointCloud(points=points, colors=colors,
                          normals=normals, is_human=is_human)

    neuman_ply_path = os.path.join(path, "points3D_with_human.ply")
    storePly(neuman_ply_path, points, (colors * 255).astype(np.uint8))

    return SceneInfo(point_cloud=pcd,
                     train_cameras=base.train_cameras,
                     test_cameras=base.test_cameras,
                     nerf_normalization=base.nerf_normalization,
                     ply_path=neuman_ply_path)


sceneLoadTypeCallbacks = {
    "Colmap": readColmapSceneInfo,
    "Blender": readNerfSyntheticInfo,
    "DTU": readNeuSDTUInfo,
    "nerfies": readNerfiesInfo,
    "plenopticVideo": readPlenopticVideoDataset,
    "NeuMan": readNeuManSceneInfo,  # Changed — Option A: NeuMan reader
}
```

**Theory.**

- **Why reuse the COLMAP reader?** NeuMan's `sparse/` directory *is* a
  standard COLMAP reconstruction. Calling `readColmapSceneInfo` first
  gives us free parsing of cameras + SfM points. The only extra step is
  appending the human seeds.
- **Why is the COLMAP cloud tagged `False`?** SfM reconstructs *static
  structure* by matching features across views. A moving subject
  violates multi-view consistency and is almost never reconstructed — so
  COLMAP points land on background with very high probability. A
  handful of bleed-through human points is tolerable and gets
  overwritten by the denser human seeds we append.
- **The traceability PLY `points3D_with_human.ply`** is written purely
  for debugging (you can open it in MeshLab to see the seed positions).
  The `is_human` flag lives in memory inside the `BasicPointCloud`; the
  *training-checkpoint* PLYs persist it separately (see §5).
- **Callback registration.** `scene/__init__.py` looks up
  `sceneLoadTypeCallbacks["NeuMan"]` — the new entry is what turns the
  autodetect branch into an actual call.


---
## 3. `scene/__init__.py` — Dispatch NeuMan scenes to the new reader

### 3.1 Goal
Route scenes to the Option A reader when either the `--is_neuman` flag is
set or when NeuMan's extra folders (`segmentations/`, `depth_maps/`) sit
next to `sparse/`. All other scene types behave exactly as before.

### 3.2 Code (lines 45–58)

```python
# Changed — Option A: dispatch to the NeuMan reader when the user sets
# --is_neuman, or auto-detect by the presence of segmentations/ +
# depth_maps/ next to the COLMAP sparse/ folder. Falls back to the
# vanilla Colmap reader otherwise. The reader itself also handles the
# sparse/0 vs sparse/ path difference.
is_neuman_arg = getattr(args, "is_neuman", False)
neuman_autodetect = (
    os.path.exists(os.path.join(args.source_path, "sparse")) and
    os.path.isdir(os.path.join(args.source_path, "segmentations")) and
    os.path.isdir(os.path.join(args.source_path, "depth_maps"))
)
if is_neuman_arg or neuman_autodetect:
    print("Assuming NeuMan data set (Option A: human-seed init).")
    scene_info = sceneLoadTypeCallbacks["NeuMan"](args.source_path, args.images, args.eval)
elif os.path.exists(os.path.join(args.source_path, "sparse")):
    scene_info = sceneLoadTypeCallbacks["Colmap"](args.source_path, args.images, args.eval)
...
```

### 3.3 Theory
- **Explicit flag vs. autodetect.** Both are provided because users who
  apply extra pre-processing (e.g. copy only COLMAP into a new folder)
  may want to force-opt-in with `--is_neuman`. Autodetect covers the
  happy path where the raw NeuMan release is untouched.
- **`getattr(args, "is_neuman", False)`** instead of direct attribute
  access makes the code robust to argument sets that don't declare the
  flag (e.g. running old scripts that construct `args` manually).
- **Placement of the branch matters.** It sits *before* the generic
  `Colmap` branch because a NeuMan scene also has `sparse/`. Without
  this ordering, the COLMAP branch would swallow the scene and skip the
  human seeding.


---
## 4. `arguments/__init__.py` — Add the `--is_neuman` CLI flag

### 4.1 Goal
Give the user an explicit opt-in toggle that `scene/__init__.py` reads.

### 4.2 Code (line 63)

```python
class ModelParams(ParamGroup):
    def __init__(self, parser, sentinel=False):
        ...
        self.is_6dof = False
        self.is_neuman = False  # Changed — Option A: opt into NeuMan reader + human/background Gaussian split
        super().__init__(parser, "Loading Parameters", sentinel)
```

### 4.3 Theory
- The `ParamGroup` base class auto-generates argparse flags from class
  attributes. Declaring `self.is_neuman = False` magically creates a
  `--is_neuman` command-line switch with `store_true` semantics.
- The flag and the autodetect (§3) are *both* supported for different
  workflows: autodetect is easier for the common case; the explicit
  flag is safer when a script is generated programmatically.


---
## 5. `scene/gaussian_model.py` — Propagate the tag through the Gaussian lifecycle

The `GaussianModel` class is a long-lived object that grows and shrinks
as training proceeds (cloning, splitting, pruning). Every one of those
mutations has to keep the `is_human` tag aligned with the row indices of
`_xyz`, `_opacity`, etc. — otherwise a background Gaussian could end up
routed through the MLP or vice-versa. There are nine small edits; we take
them in order.


### 5.1 Non-trainable buffer in `__init__`

**Goal.** Allocate an empty bool buffer at construction so later
mutations have something to extend / mask.

**Code (lines 45–49):**

```python
self.xyz_gradient_accum = torch.empty(0)
# Changed — Option A: per-Gaussian human/background tag. Non-trainable
# bool buffer kept in lock-step with _xyz across densify/prune/save/load.
# Empty-tensor sentinel means "no tags present" (legacy behaviour:
# DeformModel will route every Gaussian through the MLP).
self._is_human = torch.empty(0, dtype=torch.bool, device="cuda")
```

**Theory.**
- A **non-trainable buffer** (not wrapped in `nn.Parameter`) because the
  tag is a discrete class, not something to optimize by gradient
  descent.
- **Empty-tensor sentinel.** Using `torch.empty(0, dtype=bool)` lets us
  distinguish "no tag information available" (`numel() == 0`) from
  "every Gaussian is background" (`numel() == N, all False`) without
  relying on `None`, which would crash PyTorch indexing operations.


### 5.2 Public accessor `get_is_human`

**Goal.** Expose the tag with defensive `None`-fallback so callers never
pass a mis-aligned buffer.

**Code (lines 85–92):**

```python
# Changed — Option A: expose the per-Gaussian human tag. Returns the bool
# buffer (shape [N]) when tags exist, or None when they don't — callers
# treat None as "no split, apply deformation to all Gaussians".
@property
def get_is_human(self):
    if self._is_human.numel() == 0 or self._is_human.shape[0] != self._xyz.shape[0]:
        return None
    return self._is_human
```

**Theory.** The shape check guards against a bug that would be very
hard to notice: if a future mutation forgets to update `_is_human`, the
property returns `None`, the deformation MLP processes *all* Gaussians,
and Option A silently degenerates to the upstream baseline. Training
still runs; you just lose the split. Better than a silent misalignment
that rearranges "who is human".


### 5.3 Absorb the tag in `create_from_pcd`

**Goal.** Copy the `is_human` array from the `BasicPointCloud` into the
GPU buffer at construction.

**Code (lines 124–135):**

```python
self.max_radii2D = torch.zeros((self.get_xyz.shape[0]), device="cuda")
# Changed — Option A: take the per-point human tag from the point cloud
# if present; otherwise leave the buffer empty to preserve the original
# "deform everything" behaviour on non-NeuMan scenes.
if getattr(pcd, "is_human", None) is not None:
    self._is_human = torch.from_numpy(
        np.asarray(pcd.is_human, dtype=bool)).to(device="cuda")
    n_h = int(self._is_human.sum().item())
    print(f"[Option A] Initialized {n_h} human / "
          f"{self._is_human.numel() - n_h} background Gaussians.")
else:
    self._is_human = torch.empty(0, dtype=torch.bool, device="cuda")
```

**Theory.**
- `getattr(pcd, "is_human", None)` again protects us from legacy
  `BasicPointCloud` objects that lack the new field.
- **GPU copy once.** The buffer lives on `cuda` because all subsequent
  masking operations happen alongside other CUDA tensors. A CPU-side
  tag would trigger implicit host-device transfers on every MLP call.


### 5.4 Extend the PLY schema (`construct_list_of_attributes`)

**Goal.** Declare an extra `is_human` property on the checkpoint PLY so
`save_ply` + `load_ply` can round-trip the tag.

**Code (lines 179–183):**

```python
# Changed — Option A: persist the per-Gaussian human tag so renders /
# resumed training keep the split. Only written when tags exist.
if self._is_human.numel() > 0:
    l.append('is_human')
return l
```

**Theory.** The PLY format is a flat columnar store: each vertex row
has the same set of scalar properties. `construct_list_of_attributes`
defines those property names; `save_ply` must emit columns in the same
order. Skipping the extra property when the buffer is empty keeps the
PLY format *byte-identical* to the upstream checkpoint for non-NeuMan
runs — so existing downstream tools aren't broken.


### 5.5 Write the column in `save_ply`

**Goal.** Emit the `is_human` float column in the same row order as the
other fields.

**Code (lines 198–206):**

```python
elements = np.empty(xyz.shape[0], dtype=dtype_full)
# Changed — Option A: append the is_human column when the buffer is
# populated so the tag round-trips through the checkpoint PLY.
if self._is_human.numel() > 0:
    is_human_col = self._is_human.detach().cpu().numpy().astype(np.float32).reshape(-1, 1)
    attributes = np.concatenate((xyz, normals, f_dc, f_rest, opacities, scale, rotation, is_human_col), axis=1)
else:
    attributes = np.concatenate((xyz, normals, f_dc, f_rest, opacities, scale, rotation), axis=1)
elements[:] = list(map(tuple, attributes))
el = PlyElement.describe(elements, 'vertex')
PlyData([el]).write(path)
```

**Theory.**
- **Why `float32` and not `uint8`?** PLY supports both, but the rest of
  the columns in this checkpoint are `float32` and all stored with
  `dtype_full`. Mixing dtypes in a single `vertex` element would
  require a more complex structured dtype. Using `float32` for a 0/1
  field wastes ~3 bytes per Gaussian — cheap.
- **Column order must match `construct_list_of_attributes`.** The
  `is_human` column goes last here, and the schema declaration appends
  `'is_human'` last too (§5.4). Mismatch would corrupt every vertex.


### 5.6 Restore the buffer in `load_ply`

**Goal.** When loading an Option A checkpoint, re-populate `_is_human`.
When loading an upstream checkpoint (no field), leave the buffer empty
so behaviour gracefully reverts.

**Code (lines 258–266):**

```python
# Changed — Option A: restore the is_human buffer if the checkpoint
# PLY was written with Option A. Missing field → leave buffer empty so
# the model falls back to legacy "deform everything" behaviour.
property_names = [p.name for p in plydata.elements[0].properties]
if "is_human" in property_names:
    is_human = np.asarray(plydata.elements[0]["is_human"]).astype(bool)
    self._is_human = torch.from_numpy(is_human).to(device="cuda")
else:
    self._is_human = torch.empty(0, dtype=torch.bool, device="cuda")

self.active_sh_degree = self.max_sh_degree
```

**Theory.** Inspecting `plydata.elements[0].properties` is the standard
way to do *optional* schema extensions: check-then-read. This keeps
forward and backward compatibility:

- Load upstream PLY with Option A code → `is_human` missing → buffer
  empty → behaviour reverts to upstream.
- Load Option A PLY with upstream code → the extra column is silently
  ignored by the vanilla loader.


### 5.7 Keep alignment in `prune_points`

**Goal.** When some Gaussians are pruned, the tag buffer must be
masked the same way as every other per-Gaussian tensor.

**Code (lines 317–320):**

```python
self.max_radii2D = self.max_radii2D[valid_points_mask]
# Changed — Option A: keep the human tag aligned with _xyz after pruning.
if self._is_human.numel() > 0:
    self._is_human = self._is_human[valid_points_mask]
```

**Theory.** `prune_points` removes Gaussians whose opacity fell below
threshold or whose screen-size is too large. Because the tag is a
per-Gaussian attribute stored in the *same row order* as `_xyz`, the
fix is exactly a boolean-mask indexing with the same mask used for the
other buffers. Forgetting this single line would offset the tag by the
number of pruned points — slowly rotating "who is human" after every
densification round.


### 5.8 Extend the buffer in `densification_postfix`

**Goal.** Provide a hook so cloned/split children can have their tag
appended in lock-step with the other new per-Gaussian tensors.

**Code (lines 348–372):**

```python
# Changed — Option A: accept new_is_human so spawned children inherit the
# parent's tag. None means "skip" (keep the legacy behaviour / empty buffer).
def densification_postfix(self, new_xyz, new_features_dc, new_features_rest, new_opacities, new_scaling,
                          new_rotation, new_is_human=None):
    d = {"xyz": new_xyz, "f_dc": new_features_dc, "f_rest": new_features_rest,
         "opacity": new_opacities, "scaling": new_scaling, "rotation": new_rotation}

    optimizable_tensors = self.cat_tensors_to_optimizer(d)
    self._xyz = optimizable_tensors["xyz"]
    ...

    self.max_radii2D = torch.zeros((self.get_xyz.shape[0]), device="cuda")
    # Changed — Option A: concatenate the is_human tag for spawned children.
    if self._is_human.numel() > 0 and new_is_human is not None:
        self._is_human = torch.cat([self._is_human, new_is_human], dim=0)
```

**Theory.** `densification_postfix` is the single bottleneck through
which *all* new Gaussians enter the model, called by both `densify_and_clone`
and `densify_and_split`. Adding the tag extension here (rather than in
each caller) removes a class of bugs where one path updates the tag and
the other doesn't.


### 5.9 Inherit tags in `densify_and_split` and `densify_and_clone`

**Goal.** Decide how children get their tag. Policy: inherit from the
parent Gaussian.

**Code — split (lines 393–399):**

```python
new_opacity = self._opacity[selected_pts_mask].repeat(N, 1)
# Changed — Option A: split children inherit the parent's human tag.
new_is_human = self._is_human[selected_pts_mask].repeat(N) \
    if self._is_human.numel() > 0 else None

self.densification_postfix(new_xyz, new_features_dc, new_features_rest, new_opacity, new_scaling, new_rotation,
                           new_is_human=new_is_human)
```

**Code — clone (lines 417–423):**

```python
new_rotation = self._rotation[selected_pts_mask]
# Changed — Option A: cloned children inherit the parent's human tag.
new_is_human = self._is_human[selected_pts_mask] \
    if self._is_human.numel() > 0 else None

self.densification_postfix(new_xyz, new_features_dc, new_features_rest, new_opacities, new_scaling,
                           new_rotation, new_is_human=new_is_human)
```

**Theory.**
- **Inherit vs re-classify.** We could re-run a backprojection test on
  each new child to check "is this still inside the human mask?". We
  don't, because (a) densification happens thousands of times during
  training — too expensive, and (b) the masks are only valid in the
  input frames, not in novel locations that densification explores.
  Inheritance is conservative and cheap.
- **Split multiplicity `N`.** `densify_and_split` produces `N` (default
  2) children per selected parent, so we `.repeat(N)` the boolean tag
  to match the replicated tensors.
- **Known trade-off.** A background Gaussian that drifts onto the
  human surface remains labeled background forever. The initialization
  seeds plenty of human points to mitigate under-coverage (see
  `_seed_human_points_from_masks`).


---
## 6. `scene/deform_model.py` — The actual Option A routing

### 6.1 Goal
Apply the deformation MLP *only* to human Gaussians. Background
Gaussians get zero `d_xyz`, `d_rotation`, `d_scaling` and remain static.

### 6.2 Code (lines 16–46)

```python
# Changed — Option A: accept a per-Gaussian human mask. When provided,
# only human Gaussians go through the deformation MLP; background
# Gaussians receive zero deltas. This implements the human/background
# split without changing the MLP architecture. Passing is_human=None or
# an all-True mask preserves the original behaviour (all Gaussians
# deformed), which is what non-NeuMan scenes want.
def step(self, xyz, time_emb, is_human=None):
    if is_human is None:
        return self.deform(xyz, time_emb)

    N = xyz.shape[0]
    human_idx = is_human.nonzero(as_tuple=False).squeeze(-1)
    n_human = int(human_idx.numel())

    # Degenerate cases: route nothing or route everything.
    if n_human == 0:
        zeros3 = torch.zeros(N, 3, device=xyz.device, dtype=xyz.dtype)
        zeros4 = torch.zeros(N, 4, device=xyz.device, dtype=xyz.dtype)
        return zeros3, zeros4, zeros3
    if n_human == N:
        return self.deform(xyz, time_emb)

    d_xyz_h, d_rot_h, d_scale_h = self.deform(xyz[human_idx], time_emb[human_idx])

    d_xyz = torch.zeros(N, d_xyz_h.shape[-1], device=xyz.device, dtype=d_xyz_h.dtype)
    d_rot = torch.zeros(N, d_rot_h.shape[-1], device=xyz.device, dtype=d_rot_h.dtype)
    d_scale = torch.zeros(N, d_scale_h.shape[-1], device=xyz.device, dtype=d_scale_h.dtype)
    d_xyz[human_idx] = d_xyz_h
    d_rot[human_idx] = d_rot_h
    d_scale[human_idx] = d_scale_h
    return d_xyz, d_rot, d_scale
```

### 6.3 Theory

- **Why this is equivalent to "set background deltas = 0".** In
  Deformable 3DGS, each Gaussian's final world pose is
  `μ_final = μ_canonical + d_xyz`. Setting `d_xyz = 0` keeps the
  Gaussian at its canonical position — i.e. truly static. The same
  applies to rotation (quaternion increment, length 4) and scaling
  (length 3 in log-space).
- **Why a mask-and-scatter rather than a multiplicative mask?**
  The MLP is expensive (MLP(x, t) with positional encoding). Passing
  only the human rows (`xyz[human_idx]`) makes the forward pass
  `n_human / N` × cheaper. Multiplying a full-batch output by a mask
  would compute deltas we throw away.
- **Degenerate cases.** `n_human == 0` avoids a zero-sized MLP call
  (some PyTorch ops break on empty tensors); `n_human == N` lets us
  skip the scatter entirely. Both branches also keep autograd happy —
  the returned zero tensors carry no gradient to the MLP, but no
  gradient is expected for background Gaussians anyway.
- **Gradient flow.** Only the `human_idx` rows participate in the loss
  (implicitly, because the renderer sees `μ + d_xyz` and the renderer
  only writes nonzero pixel gradients where Gaussians contribute). The
  MLP therefore receives gradients *only* from human Gaussians, which
  is exactly the point of Option A: don't waste capacity on static
  background.
- **Why keep the MLP architecture unchanged?** Option A is a pure
  *routing* change. A future variant could specialize the MLP (e.g. a
  smaller body for background residuals), but that breaks checkpoint
  compatibility and wasn't required for this experiment.


---
## 7. Call-site updates in `train.py`, `render.py`, `train_gui.py`

### 7.1 Goal
Propagate the new `is_human` argument at every site that invokes
`deform.step(...)` (or `timer.step(...)` in the rendering benchmarks).
No other logic changes.

### 7.2 Code — `train.py` (training loop + validation)

Line 97–99 (training loop):
```python
# Changed — Option A: route only human Gaussians through the MLP.
d_xyz, d_rotation, d_scaling = deform.step(gaussians.get_xyz.detach(), time_input + ast_noise,
                                           is_human=gaussians.get_is_human)
```

Line 219–221 (validation loop inside `training_report`):
```python
# Changed — Option A: route only human Gaussians through the MLP.
d_xyz, d_rotation, d_scaling = deform.step(xyz.detach(), time_input,
                                           is_human=scene.gaussians.get_is_human)
```

### 7.3 Code — `render.py` (seven call sites)

All seven sites follow the same pattern; the representative change is:
```python
# Changed — Option A: route only human Gaussians through the MLP.
d_xyz, d_rotation, d_scaling = deform.step(xyz.detach(), time_input, is_human=gaussians.get_is_human)
```

The timing-benchmark rendering functions use the same flag with
`timer.step(...)` instead of `deform.step(...)` — same argument.

### 7.4 Code — `train_gui.py` (two call sites)

```python
# Changed — Option A: route only human Gaussians through the MLP.
d_xyz, d_rotation, d_scaling = self.deform.step(self.gaussians.get_xyz.detach(), time_input + ast_noise,
                                                is_human=self.gaussians.get_is_human)
```

and

```python
# Changed — Option A: route only human Gaussians through the MLP.
d_xyz, d_rotation, d_scaling = self.deform.step(self.gaussians.get_xyz.detach(), time_input,
                                                is_human=self.gaussians.get_is_human)
```

### 7.5 Theory
- **`gaussians.get_is_human`** returns `None` for non-NeuMan scenes
  (see §5.2). In that case `deform.step` hits the first early-return
  branch (§6.2) and behaves exactly like upstream. This is the whole
  reason the flag has a `None` fallback: the same code path runs for
  D-NeRF, NeRFies, DTU, … with zero behaviour change.
- **Why thread the flag through `render.py`'s timing benchmarks too?**
  The speed report `timer.step(...)` measures MLP latency. Running the
  background through the MLP would overestimate the cost of the new
  design; passing `is_human` lets the benchmark reflect the reduced
  work that training actually does.
- **One-line per call site = ~13 repetitions across three files.**
  We accept the duplication because Python has no good way to inject
  a new named kwarg into every existing call without either refactoring
  the deformation interface or monkey-patching — both of which obscure
  the review.


---
## 8. Summary

Conceptually Option A does exactly three things, and every change above
belongs to one of these three buckets:

| Bucket | Files | What it adds |
|---|---|---|
| **Data ingest** | `utils/graphics_utils.py`, `scene/dataset_readers.py`, `scene/__init__.py`, `arguments/__init__.py` | Read NeuMan's mask + depth pairs, backproject them into human seed Gaussians, and tag every point with a boolean `is_human`. |
| **Lifecycle of the tag** | `scene/gaussian_model.py` | Store, mask, extend, save, and load the tag in lock-step with every other per-Gaussian tensor. |
| **Routing** | `scene/deform_model.py` (+ call sites in `train.py`, `render.py`, `train_gui.py`) | Apply the deformation MLP only to human Gaussians; background Gaussians stay canonical. |

### Things to remember when reading Option A code
- `# Changed — Option A:` markers tag every edit so `grep -n "Option A"`
  gives you the full change set at a glance.
- The design is **backwards compatible**: empty tag buffer → `get_is_human`
  returns `None` → `deform.step(..., is_human=None)` takes the original
  path → non-NeuMan scenes train unchanged.
- The design is **forwards compatible with Option B**: Option B keeps the
  same reader machinery, replaces the human/background split with
  SMPL-pose conditioning, and reuses the per-Gaussian buffer pattern
  (different semantics, same bookkeeping).
